# HDFCBANK ATM Call Backtest — Full Analysis
Team: Strike Squad  
Lead: Shubhra Arora (2024A1PS0271P)  
Course: FIN F311  
Period: Jan 2022 — Dec 2024  
Date: 27th November 2025

This notebook runs the final backtest (actual vs synthetic), computes performance metrics,
generates figures for reporting, and saves results for the final deliverables.

In [25]:
import sys
import os
from pathlib import Path
import json

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np

from src.backtest_engine.backtest_engine import BacktestEngine
from src.analysis_viz.metrics import compute_all_metrics
from src.analysis_viz.visualization import generate_all_plots

pd.options.display.max_columns = 200


In [26]:
import yaml

config_path = Path(project_root) / "config" / "config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

data_dir = Path(project_root) / "data" / "processed"
figures_dir = Path(project_root) / "slides" / "figures"
outputs_dir = Path(project_root) / "data" / "processed"

figures_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)


In [27]:
atm_path = data_dir / "atm_lookup_271P.csv"
if not atm_path.exists():
    raise FileNotFoundError(f"ATM lookup not found at {atm_path}. Please run Notebook 01B to generate it.")

atm_lookup = pd.read_csv(atm_path, parse_dates=["trade_date", "expiry_date"], dayfirst=False)

required_cols = ["trade_date", "stock_price", "atm_strike", "days_to_expiry",
                 "implied_vol", "risk_free_rate", "call_mid", "liquid"]

missing = [c for c in required_cols if c not in atm_lookup.columns]
if missing:
    raise ValueError(f"ATM lookup is missing columns: {missing}")

print("ATM lookup loaded:", atm_lookup.shape)
display(atm_lookup.head())


ATM lookup loaded: (478, 20)


,trade_date,expiry_date,days_to_expiry,stock_price,atm_strike,moneyness,call_bid,call_ask,call_mid,put_bid,put_ask,put_mid,implied_vol,open_interest,volume,call_spread_pct,put_spread_pct,risk_free_rate,liquid,skip_reason
0,2023-12-15,2024-01-18,34,806.219299,810,0.995332,13.75,13.95,13.870425,12.45,12.65,12.536601,0.134515,5948,150,1.441917,1.595329,0.068,True,NaN
1,2023-12-18,2024-01-18,31,805.805542,810,0.994822,12.80,12.95,12.875233,12.30,12.50,12.405147,0.134954,5110,384,1.165027,1.612234,0.068,True,NaN
2,2023-12-19,2024-01-18,30,804.442871,800,1.005554,17.25,17.50,17.384244,8.35,8.60,8.482612,0.135855,5910,506,1.438084,2.947206,0.068,True,NaN
3,2023-12-20,2024-01-25,36,806.438232,810,0.995603,14.50,14.70,14.615347,12.65,12.85,12.762743,0.135452,5290,500,1.368425,1.567061,0.068,True,NaN
4,2023-12-21,2024-01-25,35,820.892822,820,1.001089,17.25,17.50,17.393614,11.10,11.25,11.171337,0.139222,5763,246,1.437309,1.342722,0.068,True,NaN


In [28]:
print("Days-to-expiry summary")
display(atm_lookup["days_to_expiry"].describe())

liquid_count = atm_lookup["liquid"].sum()
total = len(atm_lookup)
print(f"Liquid rows: {liquid_count} / {total} ({liquid_count/total:.1%})")


Days-to-expiry summary


count    478.000000
mean      33.148536
std        2.325888
min       30.000000
25%       31.000000
50%       34.000000
75%       35.000000
max       36.000000
Name: days_to_expiry, dtype: float64

Liquid rows: 478 / 478 (100.0%)


In [29]:
backtester = BacktestEngine()

ledger_df = backtester.run_backtest(
    atm_lookup=atm_lookup,
    output_path=str(outputs_dir / "backtest_results_271P.csv")
)

ledger_df["trade_date"] = pd.to_datetime(ledger_df["trade_date"])
ledger_df = ledger_df.sort_values("trade_date").reset_index(drop=True)

print("Backtest ledger created:", ledger_df.shape)
display(ledger_df.head())


Backtest ledger created: (478, 10)


,trade_date,stock_price,strike,dte,implied_vol,risk_free_rate,actual_price,synthetic_price,daily_pnl,cumulative_pnl
0,2023-12-15,806.219299,810,34,0.134515,0.068,13.870425,13.870425,-1.776357e-15,-1.776357e-15
1,2023-12-18,805.805542,810,31,0.134954,0.068,12.875233,12.875233,-1.776357e-15,-3.552714e-15
2,2023-12-19,804.442871,800,30,0.135855,0.068,17.384244,17.384244,0.000000e+00,-3.552714e-15
3,2023-12-20,806.438232,810,36,0.135452,0.068,14.615347,14.615347,0.000000e+00,-3.552714e-15
4,2023-12-21,820.892822,820,35,0.139222,0.068,17.393614,17.393614,0.000000e+00,-3.552714e-15


In [30]:
# Basic checks
if ledger_df["actual_price"].isnull().any():
    print("Warning: Null values found in actual prices.")

ledger_df.to_csv(outputs_dir / "backtest_ledger_271P.csv", index=False)
print("Ledger saved to:", str(outputs_dir / "backtest_ledger_271P.csv"))


Ledger saved to: /Users/shubhra/hdfcbank-strike-squad/data/processed/backtest_ledger_271P.csv


In [31]:
metrics = compute_all_metrics(ledger_df)
print("Performance metrics (dictionary):")
print(json.dumps(metrics, indent=2))
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(outputs_dir / "backtest_metrics_271P.csv", index=False)
print("Metrics saved to:", str(outputs_dir / "backtest_metrics_271P.csv"))
display(metrics_df.T)


Performance metrics (dictionary):
{
  "cumulative_return": -3.730349362740526e-14,
  "annualized_return": -1.9666276975117416e-14,
  "annualized_volatility": 1.6233276778779093e-14,
  "sharpe_ratio": -1.211479188282313,
  "max_drawdown": -4.973799150320701e-14,
  "hit_ratio": 0.0397489539748954,
  "mean_abs_pricing_error": 3.2331180968163555e-16
}
Metrics saved to: /Users/shubhra/hdfcbank-strike-squad/data/processed/backtest_metrics_271P.csv


,0
cumulative_return,-3.730349e-14
annualized_return,-1.966628e-14
annualized_volatility,1.623328e-14
sharpe_ratio,-1.211479e+00
max_drawdown,-4.973799e-14
hit_ratio,3.974895e-02
mean_abs_pricing_error,3.233118e-16


In [32]:
generate_all_plots(ledger_df, output_dir=str(figures_dir))
print("Plots saved in:", str(figures_dir))


Plots saved in: /Users/shubhra/hdfcbank-strike-squad/slides/figures


In [33]:
conv_path = outputs_dir / "convergence_results_271P.csv"
if conv_path.exists():
    conv_df = pd.read_csv(conv_path)
    display(conv_df.head())
    # Summary: find min n where absolute_error < tolerance (if in config)
    tol = config.get("pricing", {}).get("convergence_tolerance", 0.5)
    summary = conv_df.groupby("scenario").apply(
        lambda df: df[df["absolute_error"] <= tol]["n_steps"].min()
    ).reset_index().rename(columns={0: "min_n_to_meet_tol"})
    print("Convergence summary (min n steps to reach tolerance):")
    display(summary)
else:
    print("Convergence results not found at", conv_path)


Convergence results not found at /Users/shubhra/hdfcbank-strike-squad/data/processed/convergence_results_271P.csv


In [34]:
# Sensitivity: scale implied vol up/down and recompute synthetic P/L for a quick check
scales = [0.9, 1.0, 1.1]
sensitivity_results = []

for s in scales:
    temp = atm_lookup.copy()
    temp["implied_vol"] = temp["implied_vol"] * s
    ledger_temp = backtester.run_backtest(atm_lookup=temp, output_path=None)
    metrics_temp = compute_all_metrics(ledger_temp)
    metrics_temp["vol_scale"] = s
    sensitivity_results.append(metrics_temp)

sens_df = pd.DataFrame(sensitivity_results)
display(sens_df)
sens_df.to_csv(outputs_dir / "sensitivity_vol_scale_271P.csv", index=False)
print("Sensitivity results saved to:", outputs_dir / "sensitivity_vol_scale_271P.csv")


,cumulative_return,annualized_return,annualized_volatility,sharpe_ratio,max_drawdown,hit_ratio,mean_abs_pricing_error,vol_scale
0,-8.957230e+02,-4.722222e+02,8.093926e+00,-58.342786,-8.944051e+02,0.000000,1.873898e+00,0.9
1,-3.730349e-14,-1.966628e-14,1.623328e-14,-1.211479,-4.973799e-14,0.039749,3.233118e-16,1.0
2,8.972298e+02,4.730165e+02,8.074667e+00,58.580312,0.000000e+00,1.000000,1.877050e+00,1.1


Sensitivity results saved to: /Users/shubhra/hdfcbank-strike-squad/data/processed/sensitivity_vol_scale_271P.csv


In [35]:
# Ensure all key outputs are saved for submission
saved_files = {
    "ledger": outputs_dir / "backtest_ledger_271P.csv",
    "metrics": outputs_dir / "backtest_metrics_271P.csv",
    "figures_dir": figures_dir
}

for name, p in saved_files.items():
    print(f"{name} -> {p}")


ledger -> /Users/shubhra/hdfcbank-strike-squad/data/processed/backtest_ledger_271P.csv
metrics -> /Users/shubhra/hdfcbank-strike-squad/data/processed/backtest_metrics_271P.csv
figures_dir -> /Users/shubhra/hdfcbank-strike-squad/slides/figures


## Key Findings (summary)

- Cumulative P&L: `replace with value from metrics`  
- Annualized return: `replace with value from metrics`  
- Annualized volatility: `replace with value from metrics`  
- Sharpe ratio: `replace with value from metrics`  
- Max drawdown: `replace with value from metrics`  

Interpretation notes:
- Compare synthetic vs actual price behavior and explain reasons for consistent bias (if any).
- Note any periods where model and market diverge (e.g., earnings/dividend periods, extremely high realized volatility).
- Record sensitivity results and their implications for model robustness.


## Validation checklist and next steps

1. Confirm ATM lookup was generated from real or synthetic option chain and document decision in METHODOLOGY.md.  
2. Cross-check a sample of days manually: verify synthetic price computation by plugging values into Black-Scholes formula.  
3. Run additional sensitivity tests: transaction costs, alternative volatility surfaces, and binomial-based synthetic pricing.  
4. Prepare final slides using figures in `slides/figures/`. Include methodology, key results, sensitivity checks, and limitations.
